In [5]:
print("Hello World")

Hello World


In [6]:
from dotenv import load_dotenv
load_dotenv()


True

In [7]:
import pandas as pd

# QA
inputs = [
    "What are the three main paths for using AI agents in Google Cloud's agent ecosystem?",
    "What is the Agent Development Kit mainly used for?",
    "Why does grounding matter in agentic systems?"
]

# extra Questions from the report:
    # "What are the three steps in the ReAct orchestration loop?",
    # "What is the role of a vector database in RAG?"

# extra outputs from the report:
    # "The three steps are Reason, Act, and Observe.",
    # "A vector database stores and searches embeddings so the system can find information by meaning rather than exact keywords."

outputs = [
    "The three main paths are: build your own agents, use Google Cloud agents, and bring in partner agents.",
    "The Agent Development Kit is used to build, manage, evaluate, and deploy custom AI-powered agents with a code-first approach.",
    "Grounding helps agents provide accurate and trustworthy answers by connecting them to verifiable facts instead of relying only on the model's pre-trained knowledge."
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline/data/goldens.csv"
df.to_csv(csv_path, index=False)

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "rag_eval_dataset"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for statup_pdf",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

In [3]:
import sys
sys.path.append("C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:/Users/Safnas/OneDrive/Desktop/Proj/llmops-pipeline/LLMOps-pipeline/data/startup_technical_guide_ai_agents_final.pdf",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}


In [4]:
# Test the function with a sample question
test_input = {"question": "What are the three steps in the ReAct orchestration loop?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])


{"timestamp": "2026-05-19T18:32:39.496501Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-19T18:32:39.503612Z", "level": "info", "event": "YAML config loaded"}
{"key": "GROQ_API_KEY", "env_key": "GROQ_API_KEY", "timestamp": "2026-05-19T18:32:39.504593Z", "level": "info", "event": "Loaded API key from environment"}
{"keys": {"GROQ_API_KEY": "gsk_rp..."}, "timestamp": "2026-05-19T18:32:39.505603Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20260520_000239_c9ace8ec", "temp_dir": "data\\session_20260520_000239_c9ace8ec", "faiss_dir": "faiss_index\\session_20260520_000239_c9ace8ec", "sessionized": true, "timestamp": "2026-05-19T18:32:39.509607Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "startup_technical_guide_ai_agents_final.pdf", "saved_as": "data\\session_20260520_000239_c9ace8ec\\31416672.pdf", "timestamp": "2026-05-19T18:32:39.556202Z",

Question: What are the three steps in the ReAct orchestration loop?

Answer: The ReAct loop consists of three stages: **Reason** – the model generates a thought or hypothesis; **Act** – it executes a tool or action based on that thought; and **Observe** – it evaluates the outcome and feeds it back into the next reasoning cycle.


In [5]:
from langsmith import evaluate

In [8]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")


{"timestamp": "2026-05-19T18:34:35.363884Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-19T18:34:35.370719Z", "level": "info", "event": "YAML config loaded"}
{"key": "GROQ_API_KEY", "env_key": "GROQ_API_KEY", "timestamp": "2026-05-19T18:34:35.371716Z", "level": "info", "event": "Loaded API key from environment"}
{"keys": {"GROQ_API_KEY": "gsk_rp..."}, "timestamp": "2026-05-19T18:34:35.372730Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20260520_000435_73477ddc", "temp_dir": "data\\session_20260520_000435_73477ddc", "faiss_dir": "faiss_index\\session_20260520_000435_73477ddc", "sessionized": true, "timestamp": "2026-05-19T18:34:35.375712Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "startup_technical_guide_ai_agents_final.pdf", "saved_as": "data\\session_20260520_000435_73477ddc\\ad669397.pdf", "timestamp": "2026-05-19T18:34:35.413746Z",

Testing all questions from the dataset:



{"count": 64, "timestamp": "2026-05-19T18:34:41.184956Z", "level": "info", "event": "Documents loaded"}
{"chunks": 180, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-05-19T18:34:41.211466Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-05-19T18:34:41.214468Z", "level": "info", "event": "Loading embedding model"}
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniL

Q1: What are the three main paths for using AI agents in Google Cloud's agent ecosystem?
A1: The three main paths for deploying AI agents in Google Cloud’s ecosystem are:  
1. **Cloud Run** – a managed, container‑based platform.  
2. **Vertex AI Agent Engine** – a fully managed, auto‑scaling service built for AI agents.  
3. **Google Kubernetes Engine (GKE)** – a managed Kubernetes service for granular control and portability.

--------------------------------------------------------------------------------



{"count": 64, "timestamp": "2026-05-19T18:35:18.411719Z", "level": "info", "event": "Documents loaded"}
{"chunks": 180, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-05-19T18:35:18.422016Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-05-19T18:35:18.424216Z", "level": "info", "event": "Loading embedding model"}
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniL

Q2: What is the Agent Development Kit mainly used for?
A2: The Agent Development Kit is a code‑first toolkit for building AI agents. It lets you define an agent’s tools as functions, provides testing infrastructure (e.g., pytest), and supports evaluating and packaging the agent for deployment.

--------------------------------------------------------------------------------



{"count": 64, "timestamp": "2026-05-19T18:36:01.787460Z", "level": "info", "event": "Documents loaded"}
{"chunks": 180, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-05-19T18:36:01.817385Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-05-19T18:36:01.822086Z", "level": "info", "event": "Loading embedding model"}
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniL

Q3: Why does grounding matter in agentic systems?
A3: Grounding connects an agent to real‑time, verifiable data sources, letting it retrieve and reason over ground‑truth information before answering. This ensures the agent’s responses are factually accurate and up‑to‑date, rather than relying solely on static model knowledge. In agentic systems, grounding is therefore essential for reliability and trustworthiness.

--------------------------------------------------------------------------------



# Langchain built-in evaluation

In [10]:
from langsmith import evaluate
from langchain_groq import ChatGroq
from langchain_classic.evaluation import load_evaluator


dataset_name = "rag_eval_dataset"

judge_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

qa_evaluator = load_evaluator(
    evaluator="labeled_criteria",
    criteria="correctness",
    llm=judge_llm,
)


def built_in_wrapper(inputs, outputs, reference_outputs):
    try:
        result = qa_evaluator.evaluate_strings(
            input=inputs["question"],
            prediction=outputs["answer"],
            reference=reference_outputs["answer"],
        )

        return {
            "key": "correctness",
            "score": float(result.get("score", 0)),
            "comment": result.get("reasoning", ""),
        }

    except Exception as e:
        return {
            "key": "correctness",
            "score": 0.0,
            "comment": f"Evaluation failed: {str(e)}",
        }


experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[built_in_wrapper],
    experiment_prefix="test-qa-rag-built-in",
    metadata={
        "judge_model": "openai/gpt-oss-120b",
        "evaluator": "labeled_criteria",
    },
)

print("Evaluation completed!")

View the evaluation results for experiment: 'test-qa-rag-built-in-259283e7' at:
https://smith.langchain.com/o/bc10ecba-db65-4d0a-8053-e8f05b623090/datasets/cc188055-4595-4ed0-b1ae-a7833c5045c3/compare?selectedSessions=0f72118c-c46a-4fae-8972-467b4c17c5a0




0it [00:00, ?it/s]{"timestamp": "2026-05-20T09:49:03.254679Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-20T09:49:03.275036Z", "level": "info", "event": "YAML config loaded"}
{"key": "GROQ_API_KEY", "env_key": "GROQ_API_KEY", "timestamp": "2026-05-20T09:49:03.280275Z", "level": "info", "event": "Loaded API key from environment"}
{"keys": {"GROQ_API_KEY": "gsk_rp..."}, "timestamp": "2026-05-20T09:49:03.283852Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20260520_151903_fb548a6e", "temp_dir": "data\\session_20260520_151903_fb548a6e", "faiss_dir": "faiss_index\\session_20260520_151903_fb548a6e", "sessionized": true, "timestamp": "2026-05-20T09:49:03.293892Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "startup_technical_guide_ai_agents_final.pdf", "saved_as": "data\\session_20260520_151903_fb548a6e\\20950f1f.pdf", "timestamp": "2026-05-20T

Evaluation completed!


## Custom Correctness Evaluator

Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment


In [14]:
from langsmith.schemas import Run, Example
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


def correctness_evaluator(run: Run, example: Example) -> dict:
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")

    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically, even if wording differs, mark it CORRECT.
- If the output misses key facts, introduces contradictions, or is factually incorrect, mark it INCORRECT.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """Input:
{input}

Expected Output:
{expected_output}

Actual Output:
{actual_output}

Judge only correctness.

Return exactly this format:
Reasoning: [1-2 sentence explanation]
Verdict: [CORRECT or INCORRECT]""")
    ])

    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    chain = eval_prompt | llm

    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })

        response_text = response.content.strip()

        reasoning = ""
        verdict = ""

        for line in response_text.splitlines():
            line = line.strip()

            if line.lower().startswith("reasoning:"):
                reasoning = line.split(":", 1)[1].strip()

            elif line.lower().startswith("verdict:"):
                verdict = line.split(":", 1)[1].strip().upper()

        verdict_clean = verdict.strip().upper()

        if verdict_clean == "CORRECT":
            score = 1
        elif verdict_clean == "INCORRECT":
            score = 0
        else:
            score = 0
            reasoning = f"Could not parse verdict. Raw response: {response_text}"

        return {
            "key": "correctness",
            "score": score,
            "comment": f"Reasoning: {reasoning} | Verdict: {verdict_clean}"
        }

    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "comment": f"Error during evaluation: {str(e)}"
        }

### Run Evaluation with Custom Correctness Evaluator


In [15]:
from langsmith import evaluate

evaluators = [correctness_evaluator]

dataset_name = "rag_eval_dataset"

experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator using Groq LLM-as-a-Judge",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "judge_model": "openai/gpt-oss-120b",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")

View the evaluation results for experiment: 'agenticAIReport-correctness-eval-5376ad80' at:
https://smith.langchain.com/o/bc10ecba-db65-4d0a-8053-e8f05b623090/datasets/cc188055-4595-4ed0-b1ae-a7833c5045c3/compare?selectedSessions=28c23266-9fe2-4591-95e0-bdd9c115b805




0it [00:00, ?it/s]{"timestamp": "2026-05-19T19:04:42.188654Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-19T19:04:42.196646Z", "level": "info", "event": "YAML config loaded"}
{"key": "GROQ_API_KEY", "env_key": "GROQ_API_KEY", "timestamp": "2026-05-19T19:04:42.198127Z", "level": "info", "event": "Loaded API key from environment"}
{"keys": {"GROQ_API_KEY": "gsk_rp..."}, "timestamp": "2026-05-19T19:04:42.199290Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20260520_003442_35d459e9", "temp_dir": "data\\session_20260520_003442_35d459e9", "faiss_dir": "faiss_index\\session_20260520_003442_35d459e9", "sessionized": true, "timestamp": "2026-05-19T19:04:42.203648Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "startup_technical_guide_ai_agents_final.pdf", "saved_as": "data\\session_20260520_003442_35d459e9\\cd1d6fba.pdf", "timestamp": "2026-05-19T


Evaluation completed! Check the LangSmith UI for detailed results.


### Optional: Combine Multiple Evaluators

You can use multiple evaluators together to get different perspectives on your RAG system's performance.


In [ ]:
# Example: Combine custom correctness evaluator with LangChain's built-in evaluators
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    LangChainStringEvaluator("cot_qa"),  # Chain-of-thought QA evaluator
]

# Run evaluation with multiple evaluators
# Uncomment to run:
# experiment_results_combined = evaluate(
#     answer_ai_report_question,
#     data=dataset_name,
#     evaluators=combined_evaluators,
#     experiment_prefix="agenticAIReport-multi-eval",
#     description="Evaluating RAG system with multiple evaluators",
#     metadata={
#         "variant": "RAG with FAISS",
#         "evaluators": "correctness + cot_qa",
#         "chunk_size": 1000,
#         "chunk_overlap": 200,
#         "k": 5,
#     },
# )
